# Notebook 3: Forecasting Validation
This notebook compares standard Prophet with the Prophet+LSTM hybrid forecaster in a data-scarce regime.

In [ ]:
import os, sys
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
conda_prefix = r"C:\Users\harsh\anaconda3\envs\raphael-env"
lib_bin = os.path.join(conda_prefix, "Library", "bin")
if os.path.exists(lib_bin):
    if lib_bin not in os.environ["PATH"]:
        os.environ["PATH"] = lib_bin + os.pathsep + os.environ["PATH"]
    if sys.platform == 'win32' and hasattr(os, 'add_dll_directory'):
        try: os.add_dll_directory(lib_bin)
        except: pass
import torch
import sqlite3
sys.path.insert(0, os.path.abspath('..'))  # backend root
DB_PATH = os.path.abspath('../data/raphael.db')
if not os.path.exists(DB_PATH):
    DB_PATH = os.path.abspath('data/raphael.db')
assert os.path.exists(DB_PATH), f"DB not found: {DB_PATH}"
conn = sqlite3.connect(DB_PATH)
print(f"Connected to: {DB_PATH}")

PUNE_REGION_ID = conn.execute(
    "SELECT id FROM regions WHERE name='Pune Metropolitan Region'"
).fetchone()[0]
print(f"Pune region ID: {PUNE_REGION_ID}")


In [ ]:
import os, sys
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
conda_prefix = r"C:\Users\harsh\anaconda3\envs\raphael-env"
lib_bin = os.path.join(conda_prefix, "Library", "bin")
if os.path.exists(lib_bin):
    if lib_bin not in os.environ["PATH"]:
        os.environ["PATH"] = lib_bin + os.pathsep + os.environ["PATH"]
    if sys.platform == 'win32' and hasattr(os, 'add_dll_directory'):
        try: os.add_dll_directory(lib_bin)
        except: pass

import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor':   '#161b22',
    'text.color':       '#c9d1d9',
    'axes.labelcolor':  '#c9d1d9',
    'xtick.color':      '#c9d1d9',
    'ytick.color':      '#c9d1d9',
    'axes.edgecolor':   '#30363d',
    'grid.color':       '#21262d',
    'savefig.facecolor':'#0d1117',
})
ACCENT = '#58a6ff'
WARN   = '#d29922'
DANGER = '#f85149'
OK     = '#3fb950'
MUTED  = '#8b949e'

ZONES = {
    'Hadapsar Industrial':  (18.5018, 73.9320),
    'Pune NE Quadrant':     (18.5632, 73.9401),
    'Kothrud Residential':  (18.5074, 73.8077),
    'Katraj Hills':         (18.4524, 73.8567),
    'Shivajinagar':         (18.5308, 73.8474),
    'Aundh':                (18.5590, 73.8080),
}


## Section 1 — Data Availability

In [ ]:
import pandas as pd

df_aq = pd.read_sql_query("""
    SELECT station_id, station_name, observed_at, value
    FROM raw_observations
    WHERE region_id = :region_id
      AND layer_type = 'aq'
      AND value > 0 AND value < 500
    ORDER BY observed_at
""", conn, params={"region_id": PUNE_REGION_ID})
df_aq['observed_at'] = pd.to_datetime(df_aq['observed_at'])

station_stats = df_aq.groupby('station_name').agg(
    total_obs=('value', 'count'),
    min_date=('observed_at', 'min'),
    max_date=('observed_at', 'max')
).reset_index()

print("=== Station Statistics ===")
print(station_stats.to_markdown(index=False))


Prophet requires ~30+ observations for reliable seasonal 
decomposition [Taylor & Letham 2018]. With 12 days of data,
we evaluate performance in a data-scarce regime typical of 
urban monitoring deployments in developing regions.

We compare: Prophet-only vs Prophet+LSTM hybrid [Milli et al. 
2025]. Literature benchmark: Hasnain et al. (2022) report 
MAE ≈ 8.2 μg/m³ for multi-station PM2.5 forecasting.

## Section 2 — Zone Aggregation

In [ ]:
from geopy.distance import geodesic

STATION_COORDS = {
    'Savitribai Phule Pune University': (18.5308, 73.8473),
    'Hadapsar': (18.4983, 73.9258),
    'Katraj Dairy': (18.4500, 73.8650),
}

def get_nearest_zone(st_name):
    st_coord = STATION_COORDS.get(st_name, (18.5308, 73.8473))
    best_zone = None
    min_dist = float('inf')
    for zone, coord in ZONES.items():
        dist = geodesic(st_coord, coord).km
        if dist < min_dist:
            min_dist = dist
            best_zone = zone
    return best_zone

df_aq['zone'] = df_aq['station_name'].apply(get_nearest_zone)

zone_series = {}
print("=== Zone Statistics ===")
for zone in ZONES:
    sub = df_aq[df_aq['zone'] == zone]
    if len(sub) > 0:
        hourly = sub.set_index('observed_at').resample('h')['value'].mean().interpolate().reset_index()
        zone_series[zone] = hourly
        print(f"{zone}: {len(hourly)} points, {hourly['observed_at'].min()} to {hourly['observed_at'].max()} (Suitable: {'Yes' if len(hourly) >= 20 else 'No'})")
    else:
        print(f"{zone}: 0 points (Suitable: No)")


## Section 3 — Train/Test Split & Forecasting

In [ ]:
import torch
from prophet import Prophet
import torch.nn as nn
import numpy as np
import pickle
import logging
logging.getLogger('prophet').setLevel(logging.WARNING)

class ResidualLSTM(nn.Module):
    def __init__(self, hidden_size=64, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=hidden_size, num_layers=num_layers, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden_size, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

results = []
horizon_hours = 48

for zone, df_z in zone_series.items():
    if len(df_z) < 20:
        print(f"{zone} - Insufficient data ({len(df_z)} pts). Skipping.")
        continue
        
    split_idx = len(df_z) - horizon_hours
    train_data = df_z.iloc[:split_idx].copy()
    test_data = df_z.iloc[split_idx:].copy()
    
    train_data = train_data.rename(columns={'observed_at': 'ds', 'value': 'y'})
    test_data = test_data.rename(columns={'observed_at': 'ds', 'value': 'y'})
    
    # Prophet Only
    m = Prophet(daily_seasonality=True, weekly_seasonality=False, yearly_seasonality=False, changepoint_prior_scale=0.3)
    m.fit(train_data[['ds', 'y']])
    
    future = m.make_future_dataframe(periods=horizon_hours, freq='h')
    prophet_forecast = m.predict(future)
    prophet_test_pred = prophet_forecast.iloc[split_idx:]['yhat'].values
    
    # Compute training residuals
    train_pred = m.predict(train_data[['ds']])
    residuals = (train_data['y'].values - train_pred['yhat'].values).astype(np.float32)
    
    # Prophet+LSTM Hybrid
    hybrid_preds = prophet_test_pred.copy()
    
    if len(residuals) >= 24:
        LOOKBACK = 12
        X_seq, y_seq = [], []
        for i in range(LOOKBACK, len(residuals)):
            X_seq.append(residuals[i-LOOKBACK:i])
            y_seq.append(residuals[i])
        X_t = torch.FloatTensor(X_seq).unsqueeze(-1)
        y_t = torch.FloatTensor(y_seq).unsqueeze(-1)
        
        model = ResidualLSTM()
        optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
        criterion = nn.MSELoss()
        
        for epoch in range(50):
            model.train()
            optimizer.zero_grad()
            out = model(X_t)
            loss = criterion(out, y_t)
            loss.backward()
            optimizer.step()
            
        model.eval()
        last_res = residuals[-LOOKBACK:].tolist()
        lstm_res = []
        with torch.no_grad():
            for _ in range(horizon_hours):
                seq = torch.FloatTensor(last_res[-LOOKBACK:]).unsqueeze(0).unsqueeze(-1)
                pred_res = model(seq).item()
                lstm_res.append(pred_res)
                last_res.append(pred_res)
                
        hybrid_preds = prophet_test_pred + np.array(lstm_res)
        hybrid_preds = np.clip(hybrid_preds, 0, 500)
        
        os.makedirs('notebooks/models', exist_ok=True)
        with open(f'notebooks/models/prophet_{zone[:8]}.pkl', 'wb') as f:
            pickle.dump(m, f)
        torch.save(model.state_dict(), f'notebooks/models/lstm_residual_{zone[:8]}.pt')
        
    # Metrics
    actual = test_data['y'].values
    prophet_mae = np.mean(np.abs(actual - prophet_test_pred))
    prophet_rmse = np.sqrt(np.mean((actual - prophet_test_pred)**2))
    
    hybrid_mae = np.mean(np.abs(actual - hybrid_preds))
    hybrid_rmse = np.sqrt(np.mean((actual - hybrid_preds)**2))
    
    results.append({
        'Zone': zone,
        'Prophet MAE': prophet_mae,
        'Prophet RMSE': prophet_rmse,
        'P+LSTM MAE': hybrid_mae,
        'P+LSTM RMSE': hybrid_rmse,
        'Hasnain(2022)': 8.2
    })
    
    # Plot single zone
    plt.figure(figsize=(10, 5))
    plt.plot(test_data['ds'], actual, color=MUTED, label='Actual PM2.5')
    plt.plot(test_data['ds'], prophet_test_pred, color=WARN, linestyle='--', label='Prophet Only')
    plt.plot(test_data['ds'], hybrid_preds, color=ACCENT, label='Prophet+LSTM Hybrid')
    plt.title(f'{zone} 48-Hour Forecast Comparison', color='#c9d1d9')
    plt.xlabel('Timestamp')
    plt.ylabel('PM2.5 (μg/m³)')
    plt.legend()
    plt.grid(True)
    zone_fn = zone.replace(' ', '_').lower()
    plt.savefig(f'outputs/03_forecast_{zone_fn}.png')
    plt.show()


## Section 3b — Comparison of Forecasters

In [ ]:
res_df = pd.DataFrame(results)
print("=== Table: Performance Metrics by Zone ===")
print(res_df.to_markdown(index=False))


Hasnain et al. (2022) report MAE ≈ 8.2 μg/m³ for multi-city 
PM2.5 forecasting using Prophet with multiple stations 
and several months of data. Our results at 12 days should 
be compared cautiously — data volume is ~10× smaller.

## Section 4 — Aggregated Visualizations

In [ ]:
plt.figure(figsize=(10, 5))
x = np.arange(len(res_df))
width = 0.35
plt.bar(x - width/2, res_df['Prophet MAE'], width, label='Prophet Only', color=WARN)
plt.bar(x + width/2, res_df['P+LSTM MAE'], width, label='Prophet+LSTM Hybrid', color=ACCENT)
plt.axhline(8.2, color=DANGER, linestyle='--', label='Hasnain (2022) Benchmark')
plt.xticks(x, res_df['Zone'], rotation=15)
plt.ylabel('MAE (μg/m³)')
plt.title('MAE Comparison across Zones', color='#c9d1d9')
plt.legend()
plt.grid(axis='y')
plt.savefig('outputs/03_mae_comparison.png')
plt.show()


## Section 5 — 48h Forward Forecast

In [ ]:
best_zone = res_df.loc[res_df['P+LSTM MAE'].idxmin()]['Zone']
print(f"Best performing zone: {best_zone}")

df_best = zone_series[best_zone]
df_best = df_best.rename(columns={'observed_at': 'ds', 'value': 'y'})

m_full = Prophet(daily_seasonality=True, weekly_seasonality=False, yearly_seasonality=False, changepoint_prior_scale=0.3)
m_full.fit(df_best[['ds', 'y']])
future_full = m_full.make_future_dataframe(periods=48, freq='h')
forecast_full = m_full.predict(future_full)

plt.figure(figsize=(12, 6))
plt.plot(df_best['ds'], df_best['y'], color=MUTED, label='Historical PM2.5')
plt.plot(forecast_full.iloc[-48:]['ds'], forecast_full.iloc[-48:]['yhat'], color=ACCENT, label='Forecast')
plt.fill_between(forecast_full.iloc[-48:]['ds'], forecast_full.iloc[-48:]['yhat_lower'], forecast_full.iloc[-48:]['yhat_upper'], color=ACCENT, alpha=0.2, label='80% Uncertainty Band')
plt.title(f'48-Hour Forward Forecast for {best_zone}', color='#c9d1d9')
plt.xlabel('Timestamp')
plt.ylabel('PM2.5 (μg/m³)')
plt.legend()
plt.grid(True)
plt.savefig('outputs/03_48h_forecast.png')
plt.show()


## Section 6 — Summary Cell

In [ ]:
print("=== Summary of Forecasting Validation ===")
print(res_df.to_markdown(index=False))
print(f"\nAverage Prophet MAE: {res_df['Prophet MAE'].mean():.2f} μg/m³")
print(f"Average Prophet+LSTM MAE: {res_df['P+LSTM MAE'].mean():.2f} μg/m³")
print(f"Average improvement: {(res_df['Prophet MAE'].mean() - res_df['P+LSTM MAE'].mean()) / res_df['Prophet MAE'].mean() * 100:.1f}%")
conn.close()
